# Data Description
- Author: 詹明翰、謝子詮、曾翊涵（共同作者；不推定個別分工）
- Date first created: 20261126
- Background: 唯一新增衍生資料表，供固定快照原始分區五欄many-to-one對照；保留所有15518原列的分類或待確認狀態，不含幾何、面積或逐里特徵。
- Unit of observation: 一組圖層＋代碼＋名稱＋說明＋簡稱原始複合鍵；296列12欄，260單碼不足以作唯一鍵。
- Sample period: 原都市計畫固定快照；生效時點／2024–2025歷史適用未知。
- Known issues: 研究歸類需審查；68筆修復結果非面殘留、跨類重疊及覆蓋品質另見處理Notebook。3範圍原列非分區，1陽明山公園空名稱待確認；mixed_special_use不擅自拆住商工。不是所有數值為正式面積分析ready。
- Definition for each variable:
    - `source_layer_code`：原圖層字串，不當全局唯一分區
    - `original_zoning_code`：原分區代碼，單碼不唯一
    - `original_zoning_name`：原使用分區名稱；缺值保存空字串
    - `original_zoning_description`：原分區說明；保留換行與附註
    - `original_zoning_abbreviation`：原分區簡稱；五欄複合鍵一部分
    - `research_zoning_category`：13類研究操作類別，非官方全部原分類
    - `eligibility_status`：eligible_zoning／non_zoning_boundary／ambiguous_layer；other_confirmed_category為可用狀態但本次無此筆
    - `classification_method`：explicit_reviewed_compound_key_v1固定白名單
    - `verification_status`：official_supported／research_classification／ambiguous／excluded_non_zoning，並非所有代碼已官方認證
    - `source_evidence`：逐鍵官方依據URL與證據範圍或待確認原因
    - `original_record_count`：此鍵對應原始列數整數，非唯一面積或事故數
    - `notes`：研究合併規則、特殊限制與待審註記
- Data path: `data/processed/urban_zoning_crosswalk.csv`
- Source data: `data/raw/細計-面.shp/.shx/.dbf`、`data/raw/unit14_urban_planning_evidence.zip`、`data/raw/unit15_zoning_classification_evidence.zip`；精確逐檔hash見讀檔綁定。
- Processing code: [geometry validation](../code/taipei_urban_planning_geometry_validation.ipynb)
- Role: crosswalk（非analysis/train/test；不含模型資料）
- Verification status: 保存CSV獨立讀回之schema、鍵、raw覆蓋、狀態與計數；以下輸出為實際檢查。
- SHA-256: `4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f`
- Downstream analysis input: `data/processed/urban_zoning_crosswalk.csv`（須同一版本，分類／面積規則經審查後才可用於B2）


In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,io,zipfile,sys,os,platform
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
def root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'AGENTS.md').is_file() and (p/'templates').is_dir():return p
    raise FileNotFoundError('Open within the student project')
ROOT=root();TEMP=ROOT/'data/temp/unit15_validation';TEMP.mkdir(parents=True,exist_ok=True)
assert Path(sys.executable).resolve()==(ROOT/'.venv/Scripts/python.exe').resolve()
os.environ['PROJ_NETWORK']='OFF'
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def verify(bindings):
    rows=[]
    for relative,h in bindings.items():
        p=(ROOT/relative).resolve()
        assert p.is_relative_to((ROOT/'data').resolve()) and p.is_file(),relative
        assert sha(p)==h,'Input version changed: '+relative
        rows.append(dict(path=relative,bytes=p.stat().st_size,sha256=h))
    return pd.DataFrame(rows)
def show(title,frame,block=18):
    display(Markdown('**'+title+'**'))
    if len(frame)==0:display(Markdown('無符合紀錄。'))
    for start in range(0,len(frame),block):display(frame.iloc[start:start+block])
pd.set_option('display.max_colwidth',110)
pd.set_option('display.max_columns',22)
print({'python':platform.python_version(),'venv':True,'validation_date':'2026-11-26','network_in_kernel':False})


{'python': '3.12.2', 'venv': True, 'validation_date': '2026-11-26', 'network_in_kernel': False}


## Change for Your Own Data
固定CSV與原始屬性，獨立讀取驗證，不執行生產Notebook或改寫任何資料。


In [2]:
BINDINGS={'data/processed/urban_zoning_crosswalk.csv': '4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f', 'data/raw/細計-面.shp': '67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee', 'data/raw/細計-面.shx': '277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69', 'data/raw/細計-面.dbf': '29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4', 'data/raw/unit14_urban_planning_evidence.zip': '695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730', 'data/raw/unit15_zoning_classification_evidence.zip': '82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9'}
display(verify(BINDINGS))
df=pd.read_csv(ROOT/'data/processed/urban_zoning_crosswalk.csv',encoding='utf-8-sig',dtype=str,keep_default_na=False)


,path,bytes,sha256
0,data/processed/urban_zoning_crosswalk.csv,117968,4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f
1,data/raw/細計-面.shp,7556848,67d2ecc469383459e43f2609cf384a4129e09b27c590cbbb82dfa750eba758ee
2,data/raw/細計-面.shx,124244,277e5f623c5c81865aa7288fac979a279b340899a4b3bc4335638f797fc5fd69
3,data/raw/細計-面.dbf,13982136,29b0484feef66c6c8417821ada8a28ed1edc0efcdffb492476fe7ada8739e1d4
4,data/raw/unit14_urban_planning_evidence.zip,6177243,695e15f7230c076b2b827f72b4337273ff795f6fc7e9c52680641c972dd72730
5,data/raw/unit15_zoning_classification_evidence.zip,285427,82effe1535fc7577d73accf81ee0663daa78944a11517b9e50c7bc2e197ba9b9


## Summary
### Sample


In [3]:
display(df.head(10))


,source_layer_code,original_zoning_code,original_zoning_name,original_zoning_description,original_zoning_abbreviation,research_zoning_category,eligibility_status,classification_method,verification_status,source_evidence,original_record_count,notes
0,1,R1,第一種住宅區,,住1,residential,eligible_zoning,explicit_reviewed_compound_key_v1,official_supported,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,107,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
1,1,R1A,第一種住宅區(特),,住1(特),residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,5,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
2,10,R,住宅區,,住,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,156,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
3,10,R5,特定休閒旅館住宅專用區,,休閒,mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,25,特定專用、混合或聯合開發等獨立類；即使名稱含住商工，不直接併入五主類，個案細則未逐案認證。
4,10,RA,第壹類住宅用地,,住1類,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,3,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
5,10,RAA,住宅區(特),,住(特),residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,8,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
6,10,RB,第貳類住宅用地,,住2類,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,14,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
7,10,RC,第參類住宅用地,,住3類,residential,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,18,明列住宅區／住宅用地之寬類；(特)(遷)與專案住宅仍附條件，未核定細則，不等同容許使用無差別。
8,10,RD,駐華使領館辦公暨住宅專用區,,使館住專,mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,1,特定專用、混合或聯合開發等獨立類；即使名稱含住商工，不直接併入五主類，個案細則未逐案認證。
9,10,REA,特定專用區(三)(供住宅使用),,特(三),mixed_special_use,eligible_zoning,explicit_reviewed_compound_key_v1,research_classification,https://laws.gov.taipei/Law/LawSearch/LawArticleContent/FL003962 第3條；20230804歷史版；原始名稱逐鍵研究映射；unit15證據包,19,特定專用、混合或聯合開發等獨立類；即使名稱含住商工，不直接併入五主類，個案細則未逐案認證。


### Summary Stats


In [4]:
count=df.original_record_count.astype(int)
display(count.describe().rename('original_record_count').to_frame())
display(df.dtypes.rename('dtype').to_frame())
display(df.eq('').sum().rename('empty_original_or_text_fields').to_frame())
for col in ['research_zoning_category','eligibility_status','verification_status']:
    temp=df.groupby(col).original_record_count.agg(keys='size',raw_records=lambda x:x.astype(int).sum())
    temp['key_ratio_of_296']=temp['keys']/len(df);temp['raw_ratio_of_15518']=temp.raw_records/count.sum();show(col,temp.reset_index())


,original_record_count
count,296.000000
mean,52.425676
std,308.081787
min,1.000000
25%,1.000000
50%,4.000000
75%,16.000000
max,4910.000000


,dtype
source_layer_code,str
original_zoning_code,str
original_zoning_name,str
original_zoning_description,str
original_zoning_abbreviation,str
research_zoning_category,str
eligibility_status,str
classification_method,str
verification_status,str
source_evidence,str


,empty_original_or_text_fields
source_layer_code,0
original_zoning_code,0
original_zoning_name,3
original_zoning_description,260
original_zoning_abbreviation,0
research_zoning_category,0
eligibility_status,0
classification_method,0
verification_status,0
source_evidence,0


**research_zoning_category**

,research_zoning_category,keys,raw_records,key_ratio_of_296,raw_ratio_of_15518
0,agriculture_conservation,2,285,0.006757,0.018366
1,commercial,23,2904,0.077703,0.187138
2,educational,35,425,0.118243,0.027388
3,industrial,16,217,0.054054,0.013984
4,mixed_special_use,55,540,0.185811,0.034798
5,non_zoning_boundary,2,3,0.006757,0.000193
6,other_classified,12,95,0.040541,0.006122
7,other_public_facilities,85,1134,0.287162,0.073076
8,park_green,9,1286,0.030405,0.082872
9,residential,29,7737,0.097973,0.498582


**eligibility_status**

,eligibility_status,keys,raw_records,key_ratio_of_296,raw_ratio_of_15518
0,ambiguous_layer,1,1,0.003378,0.000064
1,eligible_zoning,293,15514,0.989865,0.999742
2,non_zoning_boundary,2,3,0.006757,0.000193


**verification_status**

,verification_status,keys,raw_records,key_ratio_of_296,raw_ratio_of_15518
0,ambiguous,1,1,0.003378,0.000064
1,excluded_non_zoning,2,3,0.006757,0.000193
2,official_supported,18,8468,0.060811,0.545689
3,research_classification,275,7046,0.929054,0.454053


### Assertions


In [5]:
import geopandas as gpd
keys=['source_layer_code','original_zoning_code','original_zoning_name','original_zoning_description','original_zoning_abbreviation']
expected_columns=keys+['research_zoning_category','eligibility_status','classification_method','verification_status','source_evidence','original_record_count','notes']
assert list(df)==expected_columns and df.shape==(296,12)
assert not df.duplicated(keys).any() and df.original_zoning_code.nunique()==260
assert count.sum()==15518 and count.gt(0).all()
assert set(df.verification_status)=={'official_supported','research_classification','ambiguous','excluded_non_zoning'}
assert set(df.eligibility_status)=={'eligible_zoning','non_zoning_boundary','ambiguous_layer'}
assert df.source_evidence.ne('').all() and df.notes.ne('').all()
assert count[df.research_zoning_category.eq('non_zoning_boundary')].sum()==3
assert count[df.research_zoning_category.eq('unclassified')].sum()==1
original_keys=['圖層','分區代碼','使用分區','分區說明','分區簡稱']
raw=gpd.read_file(ROOT/'data/raw/細計-面.shp',engine='pyogrio',encoding='cp950')
actual=raw[original_keys].fillna('').rename(columns=dict(zip(original_keys,keys))).groupby(keys,dropna=False).size().rename('observed_n').reset_index()
check=df.merge(actual,on=keys,how='outer',validate='one_to_one',indicator=True)
assert check['_merge'].eq('both').all() and (check.original_record_count.astype(int)==check.observed_n).all()
joined=raw[original_keys].fillna('').rename(columns=dict(zip(original_keys,keys))).merge(df,on=keys,validate='many_to_one',how='left')
assert len(joined)==15518 and joined.research_zoning_category.notna().all()
assert raw.crs is None
print('獨立讀回通過：296鍵／12欄，260碼，15518列完整一對應，原始空值與換行保存；未清理或重跑舊Notebook。')
print('SHA256',sha(ROOT/'data/processed/urban_zoning_crosswalk.csv'))


獨立讀回通過：296鍵／12欄，260碼，15518列完整一對應，原始空值與換行保存；未清理或重跑舊Notebook。
SHA256 4326d92d1567167ca67c13dc1c083426bf6d4d789b88ed5da1feec422306383f


### Interpretation and handoff
official_supported表示明示原分類有法規語義支持，不是完整本機代碼或生效版本已官方認證；research_classification為研究合併，ambiguous保留待確認，excluded_non_zoning只排除範圍作分區面積用途，原始列未刪。分類表結構／raw覆蓋已驗證，正式修復與B2規則未審查，不能由hash吻合推論全資料已ready。學生先審Unit 1.5，再決定Unit 1.6；無逐里面積或模型輸出。
